## Notebook 1
#### Notebook for preprocessing of the data.

In [2]:
from pathlib import Path
import tensorflow as tf
import pandas as pd
import numpy as np

#Paths for input and output 

INPUT_DIR = Path(
    "/Users/erinekman/cb1020/Hud-cancer/data"
)

OUTPUT_DIR = Path(
    "/Users/erinekman/cb1020/Hud-cancer/preprocessed_data"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_FILE = INPUT_DIR / "labels.csv"

#Downloading of the metadata

df = pd.read_csv(CSV_FILE)

df = df[["filename", "diagnosis"]]

#Creating arrays & counters
images = []
labels = []

processed = 0
failed = 0

#Processing of the images
#Decoding into pixels, resizing and normalization

for _, row in df.iterrows():

    image_path = INPUT_DIR / row["filename"]

    try:

        image = tf.io.read_file(str(image_path))

        #Decoding into a [height, width, 3] tensor
        image = tf.image.decode_jpeg(
            image,
            channels=3
        )
        
        #Resize to 224*224
        image = tf.image.resize_with_pad(
            image,
            224,
            224
        )

        # Normalization, divide RBG values by 255
        image = tf.cast(
            image,
            tf.float16
        ) / 255.0

        #Convertint the image from a tensor to a numpy array
        #Add it to the list images 
        images.append(image.numpy())
        labels.append(row["diagnosis"])

        #Counter +1
        processed += 1

        if processed % 500 == 0:
            print(f"Processed {processed} images")

    except Exception as e:

        failed += 1
        print(f"Failed: {image_path.name} -> {e}")

#Convert the lists images and labels to numpy arrays

images = np.array(images, dtype=np.float16)
labels = np.array(labels)

#Save the data in images.npy and labels.npy in folder "preprocessed_data"
np.save(
    OUTPUT_DIR / "images.npy",
    images
)

np.save(
    OUTPUT_DIR / "labels.npy",
    labels
)

#Print summary
print("\nDone!")
print(f"Images processed: {processed}")
print(f"Errors: {failed}")

print("\nSaved files:")
print("images.npy")
print("labels.npy")

print("\nImage shape:")
print(images.shape)


Done!
Images processed: 100
Errors: 0

Saved files:
images.npy
labels.npy

Image shape:
(100, 224, 224, 3)
